In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss
import ast

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
# Paths — adjust for your Colab Drive structure
OLD_MODEL_PATH      = 'models/final_optimized_model/final_model'
OLD_TOKENIZER_PATH  = 'models/final_optimized_model/final_tokenizer'
DATASET_PATH        = 'cleaned_articles_labeled.csv'
SAVE_PATH           = '/content/drive/MyDrive/Minor Project/models/custom_model'

CATEGORIES = [
    'Politics', 'Technology', 'Business', 'Economy',
    'International_Affairs', 'Education', 'Health',
    'Science/Other', 'Sports', 'Entertainment'
]
NUM_LABELS  = len(CATEGORIES)   # 10

# Training hyperparameters
MAX_LEN     = 256
BATCH_SIZE  = 16
EPOCHS      = 5
LR          = 2e-5              # lower LR for warm-start (model already adapted)
WARMUP_FRAC = 0.1               # 10% of steps for LR warmup

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        text = str(row['title']).strip() + ' - ' + str(row['text']).strip()

        enc = self.tokenizer(
            text,
            max_length        = self.max_len,
            padding           = 'max_length',
            truncation        = True,
            return_tensors    = 'pt',
            return_attention_mask = True,
        )

        labels = torch.tensor(
            [float(row[cat]) for cat in CATEGORIES],
            dtype=torch.float32
        )

        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         labels,
        }

In [ ]:
def compute_metrics(logits: np.ndarray, labels: np.ndarray,
                    threshold: float = 0.5) -> dict:
    probs      = 1 / (1 + np.exp(-logits))   # sigmoid
    preds      = (probs > threshold).astype(int)

    macro_f1   = f1_score(labels, preds, average='macro',  zero_division=0)
    micro_f1   = f1_score(labels, preds, average='micro',  zero_division=0)
    h_loss     = hamming_loss(labels, preds)

    # Exact match accuracy (all labels correct for a sample)
    exact_match = np.mean(np.all(preds == labels, axis=1))

    return {
        'macro_f1':    macro_f1,
        'micro_f1':    micro_f1,
        'hamming_loss': h_loss,
        'exact_match': exact_match,
    }

In [ ]:
def load_warmstart_model(old_model_path: str, num_new_labels: int):
    """
    Loads existing fine-tuned RoBERTa, replaces the classification head.

    What happens internally:
      1. Load full old model (encoder + old 8-class head)
      2. Extract encoder weights
      3. Create new model with num_new_labels output classes
      4. Copy encoder weights into new model
      5. New classification head starts from random init (Xavier uniform)

    The encoder keeps all its learned knowledge.
    Only the final linear layer is new and untrained.
    """
    print(f"Loading existing model from: {old_model_path}")

    # Load old model to extract encoder weights
    old_model = AutoModelForSequenceClassification.from_pretrained(old_model_path)

    print(f"Old model: {old_model.config.num_labels} labels")
    print(f"New model: {num_new_labels} labels")

    # Create new model config — same architecture, different num_labels
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(old_model_path)
    config.num_labels = num_new_labels

    # Build new model with the updated config
    new_model = AutoModelForSequenceClassification.from_config(config)

    # ── Copy encoder weights ────────────────────────────────────────────────
    # RoBERTa structure: model.roberta (encoder) + model.classifier (head)
    # We copy roberta weights exactly, leave classifier as random init

    old_state  = old_model.state_dict()
    new_state  = new_model.state_dict()

    copied, skipped = 0, 0
    for key in new_state:
        if key in old_state and old_state[key].shape == new_state[key].shape:
            new_state[key] = old_state[key]
            copied += 1
        else:
            # Shape mismatch = classification head (different num_labels)
            # Leave as random init from from_config()
            skipped += 1
            print(f"  New layer (random init): {key}  "
                  f"{new_state[key].shape}")

    new_model.load_state_dict(new_state)
    print(f"\nWeight transfer: {copied} layers copied, {skipped} reinitialized")

    # Apply proper Xavier initialization to the new classification head
    for name, param in new_model.named_parameters():
        if 'classifier' in name and param.dim() >= 2:
            nn.init.xavier_uniform_(param)
            print(f"  Xavier init applied: {name}")

    del old_model   # free memory
    return new_model

In [ ]:
def train_epoch(model, loader, optimizer, scheduler,
                criterion, device, epoch, total_epochs):
    model.train()
    total_loss = 0
    all_logits, all_labels = [], []

    for step, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        all_logits.append(outputs.logits.detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

        if (step + 1) % 20 == 0:
            print(f"  Epoch {epoch}/{total_epochs} | "
                  f"Step {step+1}/{len(loader)} | "
                  f"Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    metrics  = compute_metrics(
        np.vstack(all_logits), np.vstack(all_labels)
    )
    return avg_loss, metrics

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_logits, all_labels = [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
        loss       = criterion(outputs.logits, labels)

        total_loss += loss.item()
        all_logits.append(outputs.logits.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    metrics  = compute_metrics(
        np.vstack(all_logits), np.vstack(all_labels)
    )
    return avg_loss, metrics

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
print(f"\nLoading dataset: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)

In [ ]:
if 'llm_labels' in df.columns:
        df['llm_labels'] = df['llm_labels'].apply(
            lambda x: ast.literal_eval(str(x)) if isinstance(x, str) else x
        )
        # Create binary columns
        for cat in CATEGORIES:
            df[cat] = df['llm_labels'].apply(lambda x: 1 if cat in x else 0)

In [ ]:
# Validate — drop rows with no labels
label_sums = df[CATEGORIES].sum(axis=1)
df = df[label_sums > 0].reset_index(drop=True)

print(f"Total articles: {len(df)}")
print(f"Label distribution:")
for cat in CATEGORIES:
    print(f"  {cat:<25} {df[cat].sum():>4} ({df[cat].mean()*100:.1f}%)")

In [ ]:
train_df,temp_df = train_test_split(df, test_size=0.25, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
print(f"\nSplits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

In [ ]:
test_df.to_csv(os.path.join(SAVE_PATH, 'test_set.csv'), index=False)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(OLD_TOKENIZER_PATH)

In [ ]:
train_dataset = NewsDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = NewsDataset(val_df,   tokenizer, MAX_LEN)

In [ ]:
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
model = load_warmstart_model(OLD_MODEL_PATH, NUM_LABELS)
model = model.to(device)

In [ ]:
label_counts = train_df[CATEGORIES].sum().values
pos_weights  = torch.tensor(
    (len(train_df) - label_counts) / np.maximum(label_counts, 1),
    dtype=torch.float32
).to(device)

In [ ]:
print(f"\npos_weights: {dict(zip(CATEGORIES, pos_weights.cpu().numpy().round(2)))}")
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

In [ ]:
encoder_params    = [p for n, p in model.named_parameters() if 'classifier' not in n]
classifier_params = [p for n, p in model.named_parameters() if 'classifier' in n]

In [ ]:
optimizer = torch.optim.AdamW([
        {'params': encoder_params,    'lr': LR,        'weight_decay': 0.01},
        {'params': classifier_params, 'lr': LR * 5,   'weight_decay': 0.01},
    ])

In [ ]:
from transformers import get_linear_schedule_with_warmup
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

In [ ]:
print(f"\n{'='*60}")
print(f"TRAINING — {EPOCHS} epochs, {total_steps} total steps")
print(f"{'='*60}")

best_macro_f1 = 0.0
history       = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{EPOCHS} ---")

    train_loss, train_metrics = train_epoch(
        model, train_loader, optimizer, scheduler,
        criterion, device, epoch, EPOCHS
    )

    val_loss, val_metrics = evaluate(
        model, val_loader, criterion, device
    )

    print(f"\nEpoch {epoch} summary:")
    print(f"  Train loss:  {train_loss:.4f} | "
            f"Macro F1: {train_metrics['macro_f1']:.4f}")
    print(f"  Val loss:    {val_loss:.4f}   | "
            f"Macro F1: {val_metrics['macro_f1']:.4f} | "
            f"Hamming: {val_metrics['hamming_loss']:.4f}")

    history.append({
        'epoch':           epoch,
        'train_loss':      train_loss,
        'train_macro_f1':  train_metrics['macro_f1'],
        'val_loss':        val_loss,
        'val_macro_f1':    val_metrics['macro_f1'],
        'val_micro_f1':    val_metrics['micro_f1'],
        'val_hamming':     val_metrics['hamming_loss'],
        'val_exact_match': val_metrics['exact_match'],
    })

    # Save best model
    if val_metrics['macro_f1'] > best_macro_f1:
        best_macro_f1 = val_metrics['macro_f1']
        model.save_pretrained(os.path.join(SAVE_PATH, 'best_model'))
        tokenizer.save_pretrained(os.path.join(SAVE_PATH, 'best_tokenizer'))
        print(f"  *** New best model saved (Macro F1: {best_macro_f1:.4f}) ***")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"Best Val Macro F1: {best_macro_f1:.4f}")
print(f"Model saved to:    {SAVE_PATH}/best_model")
print(f"{'='*60}")

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(SAVE_PATH, 'training_history.csv'), index=False)

In [ ]:
print(f"\nLoading best model for test evaluation...")
best_model = AutoModelForSequenceClassification.from_pretrained(
    os.path.join(SAVE_PATH, 'best_model')
).to(device)

In [ ]:
test_dataset = NewsDataset(test_df, tokenizer, MAX_LEN)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,shuffle=False, num_workers=2)

In [ ]:
_, test_metrics = evaluate(best_model, test_loader, criterion, device)

In [ ]:
print(f"\nTest set results:")
print(f"  Macro F1:    {test_metrics['macro_f1']:.4f}")
print(f"  Micro F1:    {test_metrics['micro_f1']:.4f}")
print(f"  Hamming Loss: {test_metrics['hamming_loss']:.4f}")
print(f"  Exact Match: {test_metrics['exact_match']:.4f}")

In [ ]:
with open(os.path.join(SAVE_PATH, 'test_results.json'), 'w') as f:
    json.dump(test_metrics, f, indent=2)

In [ ]:
# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    

    os.makedirs(SAVE_PATH, exist_ok=True)

    # ── Load dataset ───────────────────────────────────────────────────────────
    

    # Parse llm_labels if they exist as strings
    

    
    

    # ── Train/val/test split ───────────────────────────────────────────────────
    

    

    # Save test set separately for later evaluation
    

    # ── Tokenizer ──────────────────────────────────────────────────────────────
    
    

    # ── Model ──────────────────────────────────────────────────────────────────


    # ── Loss with pos_weight for class imbalance ───────────────────────────────


    

    # ── Optimizer — discriminative learning rates ──────────────────────────────
    # Lower LR for encoder (already trained), higher for new head
    

    

    # The new classification head gets 5x higher LR because it's randomly
    # initialized and needs to learn faster than the warm encoder

    # ── LR scheduler with warmup ───────────────────────────────────────────────
    

    # ── Training ───────────────────────────────────────────────────────────────
    

    # ── Save training history ──────────────────────────────────────────────────
    

    

    # ── Final test evaluation ──────────────────────────────────────────────────
    

   

    

    

    

    return best_macro_f1


if __name__ == '__main__':
    main()